In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm


In [2]:
companies = pd.read_excel("../data/PMData.xlsx",sheet_name="Company Meta Data")
prices = pd.read_excel("../data/PMData.xlsx", sheet_name='Adjusted Close')
prices.set_index('date', inplace=True)

In [3]:
companies

,ticker,company_name,country,sector,currency
0,VST,Vistra Corp.,United States,Utilities,USD
1,VIST,"Vista Energy, S.A.B. de C.V.",Mexico,Energy,MXN
2,AEP,"American Electric Power Company, Inc.",United States,Utilities,USD
3,ETR,Entergy Corporation,United States,Utilities,USD
4,AEP,"American Electric Power Company, Inc.",United States,Utilities,USD
...,...,...,...,...,...
74,SGD/USD,SGD/USD exchange rate,NaN,NaN,NaN
75,CAD/USD,CAD/USD exchange rate,NaN,NaN,NaN
76,DKK/USD,DKK/USD exchange rate,NaN,NaN,NaN
77,EUR/USD,EUR/USD exchange rate,NaN,NaN,NaN


In [4]:
t_weight = np.full(80, .986)
t_weight[0] = 1
t_weight_schedule = np.cumprod(t_weight)

t_weight_schedule = 80 / np.sum(t_weight_schedule) * t_weight_schedule
t_weight_schedule = t_weight_schedule.reshape((-1,1))

In [5]:
t_weight_schedule

array([[1.65608895],
       [1.6329037 ],
       [1.61004305],
       [1.58750245],
       [1.56527741],
       [1.54336353],
       [1.52175644],
       [1.50045185],
       [1.47944553],
       [1.45873329],
       [1.43831102],
       [1.41817467],
       [1.39832022],
       [1.37874374],
       [1.35944133],
       [1.34040915],
       [1.32164342],
       [1.30314041],
       [1.28489645],
       [1.2669079 ],
       [1.24917119],
       [1.23168279],
       [1.21443923],
       [1.19743708],
       [1.18067296],
       [1.16414354],
       [1.14784553],
       [1.13177569],
       [1.11593083],
       [1.1003078 ],
       [1.08490349],
       [1.06971484],
       [1.05473884],
       [1.03997249],
       [1.02541288],
       [1.0110571 ],
       [0.9969023 ],
       [0.98294567],
       [0.96918443],
       [0.95561584],
       [0.94223722],
       [0.9290459 ],
       [0.91603926],
       [0.90321471],
       [0.8905697 ],
       [0.87810173],
       [0.8658083 ],
       [0.853

In [6]:
sector_etfs = [
    "XLB", "XLC", "XLE", "XLF", "XLI", "XLK",
    "XLP", "XLRE", "XLU", "XLV", "XLY",
]


# All overlapping four-week log returns
returns_4w = np.log(prices).diff(4)


raw_4w_prices = {}
raw_4w_returns = {}
weight_vectors = {}
weighted_4w_returns = {}
sector_residuals = {}
regression_records = []


for i in range(4):
    stagger_name = f"stagger_{i + 1}"

    # 81 price endpoints produce 80 non-overlapping returns
    stagger_prices = prices.iloc[i::4].copy()

    # 80 raw four-week log returns
    stagger_returns = returns_4w.iloc[4 + i::4].copy()

    # Recent dates receive the largest weights
    weights = pd.Series(
        t_weight_schedule[::-1].ravel(),
        index=stagger_returns.index,
        name="time_weight",
    )

    # Time-weighted four-week log returns
    weighted_returns = stagger_returns.mul(
        weights,
        axis=0,
    )

    residuals = pd.DataFrame(index=weighted_returns.index)

    for sector in sector_etfs:
        regression_data = weighted_returns[
            ["SPY", sector]
        ].dropna()

        X = sm.add_constant(regression_data["SPY"])
        y = regression_data[sector]

        model = sm.OLS(y, X).fit()

        regression_records.append({
            "stagger": stagger_name,
            "sector": sector,
            "alpha": model.params["const"],
            "beta": model.params["SPY"],
            "r_squared": model.rsquared,
            "alpha_t_stat": model.tvalues["const"],
            "beta_t_stat": model.tvalues["SPY"],
            "alpha_p_value": model.pvalues["const"],
            "beta_p_value": model.pvalues["SPY"],
            "observations": int(model.nobs),
        })

        residuals.loc[
            regression_data.index,
            f"{sector}_residual",
        ] = model.resid

    raw_4w_prices[stagger_name] = stagger_prices
    raw_4w_returns[stagger_name] = stagger_returns
    weight_vectors[stagger_name] = weights
    weighted_4w_returns[stagger_name] = weighted_returns
    sector_residuals[stagger_name] = residuals


regression_stats = pd.DataFrame(regression_records)

In [7]:
output_file = Path(
    "../data/companies_and_staggered_returns.xlsx"
)

output_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)


def add_date_column(data):
    return (
        data
        .rename_axis("date")
        .reset_index()
    )


def write_section(
    writer,
    sheet_name,
    title,
    data,
    start_row,
):
    # Section title
    pd.DataFrame([[title]]).to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=start_row,
        index=False,
        header=False,
    )

    # Section table
    data.to_excel(
        writer,
        sheet_name=sheet_name,
        startrow=start_row + 1,
        index=False,
    )

    # Return the starting row for the next section
    return start_row + len(data) + 4


with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
) as writer:

    companies.to_excel(
        writer,
        sheet_name="Companies",
        index=False,
    )

    regression_stats.to_excel(
        writer,
        sheet_name="Regression Stats",
        index=False,
    )

    for i in range(1, 5):
        stagger_name = f"stagger_{i}"
        row = 0

        row = write_section(
            writer,
            stagger_name,
            "Four-week endpoint prices (81 observations)",
            add_date_column(raw_4w_prices[stagger_name]),
            row,
        )

        row = write_section(
            writer,
            stagger_name,
            "Raw four-week log returns (80 observations)",
            add_date_column(raw_4w_returns[stagger_name]),
            row,
        )

        row = write_section(
            writer,
            stagger_name,
            "Time-weight vector (80 observations)",
            (
                weight_vectors[stagger_name]
                .rename_axis("date")
                .reset_index()
            ),
            row,
        )

        row = write_section(
            writer,
            stagger_name,
            "Time-weighted four-week log returns",
            add_date_column(
                weighted_4w_returns[stagger_name]
            ),
            row,
        )

        write_section(
            writer,
            stagger_name,
            "Sector regression residual returns",
            add_date_column(
                sector_residuals[stagger_name]
            ),
            row,
        )

output_file

PosixPath('../data/companies_and_staggered_returns.xlsx')